# CH Data Inspection

Inspect ClickHouse tables: schemas, row counts, data completeness, and `market_snapshot_collector` vs `market_snapshot` comparison.

In [ ]:
import os
from datetime import datetime
from zoneinfo import ZoneInfo

import clickhouse_connect
import polars as pl

ET = ZoneInfo("America/New_York")

ch = clickhouse_connect.get_client(
    host="localhost",
    port=8123,
    username="default",
    password=os.getenv("CLICKHOUSE_PASSWORD", ""),
    database="jerry_trader",
)
print("CH connected")

In [ ]:
DATE = "2026-03-05"

## 1. Table Schemas

In [ ]:
tables = ["market_snapshot_collector", "market_snapshot", "trades", "quotes"]

for table in tables:
    try:
        cols = ch.query(f"DESCRIBE TABLE {table}").result_rows
        print(f"\n=== {table} ({len(cols)} columns) ===")
        for name, typ in [(r[0], r[1]) for r in cols]:
            print(f"  {name:30s} {typ}")
    except Exception as e:
        print(f"\n=== {table} === NOT FOUND: {e}")


=== market_snapshot_collector (16 columns) ===
  ticker                         String
  timestamp                      Int64
  price                          Float64
  volume                         Float64
  prev_close                     Float64
  prev_volume                    Float64
  vwap                           Float64
  bid                            Float64
  ask                            Float64
  bid_size                       Float64
  ask_size                       Float64
  changePercent                  Float64
  date                           String
  mode                           String
  session_id                     String
  inserted_at                    DateTime

=== market_snapshot (22 columns) ===
  symbol                         String
  date                           String
  mode                           String
  session_id                     String
  event_time_ms                  Int64
  event_time                     DateTime64(3, 'UTC')
  price   

## 2. Row Counts & Dates

In [ ]:
for table in tables:
    try:
        # Total count
        total = ch.query(f"SELECT count() FROM {table} FINAL").result_rows[0][0]
        # Dates available
        dates = ch.query(
            f"SELECT date, count() as cnt FROM {table} FINAL GROUP BY date ORDER BY date"
        ).result_rows
        print(f"\n=== {table} — {total:,} total rows ===")
        for d, cnt in dates:
            marker = " <<<" if d == DATE else ""
            print(f"  {d}: {cnt:>10,} rows{marker}")
    except Exception as e:
        print(f"\n=== {table} === {e}")


=== market_snapshot_collector — 641,839 total rows ===
  2026-03-13:    641,839 rows <<<

=== market_snapshot — 53,857 total rows ===
  2026-03-13:     53,857 rows <<<

=== trades — 4,020,272 total rows ===
  2026-03-13:  4,020,272 rows <<<

=== quotes — 2,682,695 total rows ===
  2026-03-13:  2,682,695 rows <<<


## 3. market_snapshot_collector — Detail for Date

In [ ]:
r = ch.query(
    "SELECT"
    "  count(),"
    "  count(DISTINCT ticker),"
    "  min(timestamp),"
    "  max(timestamp),"
    "  any(mode),"
    "  any(session_id) "
    "FROM market_snapshot_collector FINAL "
    "WHERE date = {date:String}",
    parameters={"date": DATE},
).result_rows[0]

cnt, tickers, ts_min, ts_max, mode, session = r
t_min = datetime.fromtimestamp(ts_min / 1000, tz=ET).strftime("%H:%M:%S")
t_max = datetime.fromtimestamp(ts_max / 1000, tz=ET).strftime("%H:%M:%S")
n_windows = len(ch.query(
    "SELECT count(DISTINCT timestamp) FROM market_snapshot_collector FINAL "
    "WHERE date = {date:String}",
    parameters={"date": DATE},
).result_rows[0])

print(f"Rows:          {cnt:,}")
print(f"Tickers:       {tickers:,}")
print(f"Windows:       {n_windows:,}")
print(f"Time range:    {t_min} - {t_max} ET")
print(f"Mode:          {mode}")
print(f"Session ID:    {session}")

Rows:          641,839
Tickers:       7,226
Windows:       1
Time range:    04:00:00 - 09:29:55 ET
Mode:          replay
Session ID:    20260313_replay


In [7]:
# Top tickers by row count (most active)
top = ch.query(
    "SELECT ticker, count() as cnt, "
    "  min(changePercent) as min_chg, max(changePercent) as max_chg, "
    "  anyLast(price) as last_price "
    "FROM market_snapshot_collector FINAL "
    "WHERE date = {date:String} "
    "GROUP BY ticker ORDER BY cnt DESC LIMIT 20",
    parameters={"date": DATE},
).result_rows

print(f"{'Ticker':10s} {'Rows':>8s} {'MinChg%':>10s} {'MaxChg%':>10s} {'LastPrice':>10s}")
print("-" * 52)
for t, cnt, mn, mx, p in top:
    print(f"{t:10s} {cnt:>8,} {mn:>10.2f} {mx:>10.2f} {p:>10.2f}")

Ticker         Rows    MinChg%    MaxChg%  LastPrice
----------------------------------------------------
EONR          3,953       5.19      44.56       1.61
ISPC          3,940      34.73     111.92       0.36
SOXL          3,936      -2.55       3.92      51.15
TQQQ          3,782      -1.86       2.18      47.11
PAYP          3,780       6.28      51.43      22.81
USO           3,643      -3.96       1.66     115.95
ELPW          3,619      26.18      61.28       5.06
SQQQ          3,602      -2.23       1.86      73.85
NVDA          3,599      -0.60       1.40     183.92
IMMP          3,557       0.00       0.00       0.61
QQQ           3,490      -0.58       0.78     598.63
TSLA          3,481      -0.33       1.38     397.82
SPY           3,387      -0.50       0.75     667.55
CXAI          3,318       5.45      51.23       0.27
VEEE          3,284      11.76      37.67       0.49
SOXS          3,265      -4.00       2.98      41.08
TSLL          3,180      -0.78       2.56     

In [8]:
# Sample rows (first window)
sample = ch.query(
    "SELECT * FROM market_snapshot_collector FINAL "
    "WHERE date = {date:String} "
    "ORDER BY timestamp ASC LIMIT 5",
    parameters={"date": DATE},
)
cols = list(sample.column_names)
df = pl.DataFrame({col: [row[i] for row in sample.result_rows] for i, col in enumerate(cols)})
print(f"Columns: {cols}")
print(f"\nNo rank/change columns? {'rank' not in cols and 'change' not in cols}")
df

Columns: ['ticker', 'timestamp', 'price', 'volume', 'prev_close', 'prev_volume', 'vwap', 'bid', 'ask', 'bid_size', 'ask_size', 'changePercent', 'date', 'mode', 'session_id', 'inserted_at']

No rank/change columns? True


ticker,timestamp,price,volume,prev_close,prev_volume,vwap,bid,ask,bid_size,ask_size,changePercent,date,mode,session_id,inserted_at
str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,"datetime[μs, Asia/Shanghai]"
"""AA""",1773388800000,66.26,126.0,65.93,8.049918e6,66.116984,66.2,66.5,100.0,100.0,0.50053,"""2026-03-13""","""replay""","""20260313_replay""",2026-04-12 17:26:06 CST
"""AAL""",1773388800000,10.6,149.0,10.55,1.12427483e8,10.581059,10.54,10.61,100.0,100.0,0.473932,"""2026-03-13""","""replay""","""20260313_replay""",2026-04-12 17:26:06 CST
"""AAOI""",1773388800000,103.073,272.0,106.190002,1.5398432e7,102.021348,102.02,103.9,500.0,100.0,-2.935307,"""2026-03-13""","""replay""","""20260313_replay""",2026-04-12 17:26:06 CST
"""AAPB""",1773388800000,27.49,36.0,27.643801,52638.0,27.49,27.4,27.53,200.0,200.0,-0.556366,"""2026-03-13""","""replay""","""20260313_replay""",2026-04-12 17:26:06 CST
"""AAPL""",1773388800000,255.1875,1829.0,255.759995,4.079402e7,255.0884,254.7,255.2,100.0,200.0,-0.223841,"""2026-03-13""","""replay""","""20260313_replay""",2026-04-12 17:26:06 CST


## 4. market_snapshot — Detail for Date

In [9]:
try:
    r = ch.query(
        "SELECT"
        "  count(),"
        "  count(DISTINCT symbol),"
        "  min(event_time_ms),"
        "  max(event_time_ms),"
        "  any(mode),"
        "  any(session_id) "
        "FROM market_snapshot FINAL "
        "WHERE date = {date:String}",
        parameters={"date": DATE},
    ).result_rows[0]

    cnt, tickers, ts_min, ts_max, mode, session = r
    t_min = datetime.fromtimestamp(ts_min / 1000, tz=ET).strftime("%H:%M:%S")
    t_max = datetime.fromtimestamp(ts_max / 1000, tz=ET).strftime("%H:%M:%S")

    print(f"Rows:          {cnt:,}")
    print(f"Tickers:       {tickers:,}")
    print(f"Time range:    {t_min} - {t_max} ET")
    print(f"Mode:          {mode}")
    print(f"Session ID:    {session}")
except Exception as e:
    print(f"No market_snapshot data: {e}")

Rows:          53,857
Tickers:       117
Time range:    04:00:00 - 09:29:55 ET
Mode:          replay
Session ID:    20260313_replay


In [10]:
try:
    sample = ch.query(
        "SELECT * FROM market_snapshot FINAL "
        "WHERE date = {date:String} "
        "ORDER BY event_time_ms ASC LIMIT 5",
        parameters={"date": DATE},
    )
    cols = list(sample.column_names)
    df = pl.DataFrame({col: [row[i] for row in sample.result_rows] for i, col in enumerate(cols)})
    print(f"Columns: {cols}")
    df
except Exception as e:
    print(f"No data: {e}")

Columns: ['symbol', 'date', 'mode', 'session_id', 'event_time_ms', 'event_time', 'price', 'changePercent', 'volume', 'prev_close', 'prev_volume', 'vwap', 'bid', 'ask', 'bid_size', 'ask_size', 'rank', 'competition_rank', 'change', 'relativeVolumeDaily', 'relativeVolume5min', 'inserted_at']


## 5. Collector vs Snapshot Comparison

In [11]:
try:
    collector_tickers = set(
        r[0] for r in ch.query(
            "SELECT DISTINCT ticker FROM market_snapshot_collector FINAL "
            "WHERE date = {date:String}",
            parameters={"date": DATE},
        ).result_rows
    )
    snapshot_tickers = set(
        r[0] for r in ch.query(
            "SELECT DISTINCT symbol FROM market_snapshot FINAL "
            "WHERE date = {date:String}",
            parameters={"date": DATE},
        ).result_rows
    )

    print(f"Collector tickers:  {len(collector_tickers):,}")
    print(f"Snapshot tickers:   {len(snapshot_tickers):,}")
    print(f"Subscription ratio: {len(snapshot_tickers)/len(collector_tickers)*100:.1f}%")
    print(f"\nIn snapshot but not collector: {snapshot_tickers - collector_tickers}")
    print(f"In collector but not snapshot: {len(collector_tickers - snapshot_tickers):,} tickers")

    # Show some collector-only tickers (filtered out by subscription logic)
    only_collector = sorted(collector_tickers - snapshot_tickers)
    if only_collector:
        print(f"\nCollector-only (first 20): {only_collector[:20]}")
except Exception as e:
    print(f"Comparison failed: {e}")

Collector tickers:  7,226
Snapshot tickers:   117
Subscription ratio: 1.6%

In snapshot but not collector: set()
In collector but not snapshot: 7,109 tickers

Collector-only (first 20): ['A', 'AA', 'AAAU', 'AAL', 'AALG', 'AAOI', 'AAON', 'AAP', 'AAPB', 'AAPD', 'AAPL', 'AAPU', 'AAPW', 'AAPX', 'AARD', 'AAUC', 'AAXJ', 'AB', 'ABAT', 'ABBV']


## 6. Data Completeness

In [12]:
# Check window gaps (expect 5s intervals)
try:
    windows = ch.query(
        "SELECT DISTINCT timestamp FROM market_snapshot_collector FINAL "
        "WHERE date = {date:String} ORDER BY timestamp ASC",
        parameters={"date": DATE},
    ).result_rows
    ts_list = [r[0] for r in windows]
    print(f"Total windows: {len(ts_list):,}")

    if len(ts_list) > 1:
        gaps = []
        for i in range(1, len(ts_list)):
            diff = ts_list[i] - ts_list[i-1]
            if diff != 5000:
                t_prev = datetime.fromtimestamp(ts_list[i-1]/1000, tz=ET).strftime("%H:%M:%S")
                t_curr = datetime.fromtimestamp(ts_list[i]/1000, tz=ET).strftime("%H:%M:%S")
                gaps.append((t_prev, t_curr, diff))
        print(f"Non-5s gaps: {len(gaps)}")
        for prev, curr, diff in gaps[:10]:
            print(f"  {prev} -> {curr}: {diff/1000:.0f}s")
        if not gaps:
            print("All windows are 5s apart — no gaps")
except Exception as e:
    print(f"Completeness check failed: {e}")

Total windows: 3,960
Non-5s gaps: 0
All windows are 5s apart — no gaps


## 7. Trades & Quotes

In [13]:
for table in ["trades", "quotes"]:
    try:
        r = ch.query(
            f"SELECT count(), count(DISTINCT ticker), "
            f"  min(sip_timestamp), max(sip_timestamp) "
            f"FROM {table} FINAL WHERE date = {{date:String}}",
            parameters={"date": DATE},
        ).result_rows[0]
        cnt, tickers, ts_min, ts_max = r
        if ts_min > 0:
            t_min = datetime.fromtimestamp(ts_min / 1e9, tz=ET).strftime("%H:%M:%S")
            t_max = datetime.fromtimestamp(ts_max / 1e9, tz=ET).strftime("%H:%M:%S")
        else:
            t_min = t_max = "N/A"
        print(f"{table}: {cnt:,} rows, {tickers} tickers, {t_min} - {t_max} ET")
    except Exception as e:
        print(f"{table}: {e}")

trades: 4,020,272 rows, 117 tickers, 04:00:00 - 20:00:00 ET
quotes: 2,682,695 rows, 117 tickers, 01:30:04 - 20:00:00 ET
